# COMP219
## Lab 4

In this week's lab, we will be looking at the k-nearest neighbours (kNN) for classification problems. The following is the kNN implementation you can find on the [GitHub page](https://github.com/xiaoweih/AISafetyLectureNotes/blob/main/Part_2/6_KNN.py):

In [1]:
import math
import copy
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
from math import sqrt
from collections import Counter
import itertools
import copy 

#load the dataset,
iris = datasets.load_iris()
X = iris.data 
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20) #training/validation data 80:20 ratio

#kNN
def kNNClassify(K, X_train, y_train, X_predict):
    distances = [sqrt(np.sum((x - X_predict)**2)) for x in X_train]
    sort = np.argsort(distances)
    topK = [y_train[i] for i in sort[:K]]
    votes = Counter(topK)
    y_predict = votes.most_common(1)[0][0]
    return y_predict

#predict the accuracy of kNN
def kNN_predict(K, X_train, y_train, X_predict, y_predict):
    acc = 0
    for i in range(len(X_predict)):
        if y_predict[i] == kNNClassify(K, X_train, y_train, X_predict[i]):
            acc += 1
    print(acc/len(X_predict))
    return acc/len(X_predict)

#display training and validation accuracies
print("Training accuracy is ", end='')
kNN_predict(3, X_train, y_train, X_train, y_train)
print("Validation accuracy is ", end='')
kNN_predict(3, X_train, y_train, X_test, y_test)

Training accuracy is 0.9833333333333333
Validation accuracy is 0.9


0.9

Once again, we are using the iris dataset. We will perform a simple membership inference attack (MIA) on our model. We split the iris dataset into two subsets: a training subset and a test (validation) subset with an 80/20 ratio (line 16 in the block of code above). Our task is to perform MIA to determine whether an attacker can infer if a specific sample was used in the training subset or validation subset of the data. 

In a more complex scenario, MIA would be used to assess the robustness and privacy of a kNN (or any other machine learning) model.

In [2]:
#MIA
def mia(K, X_train, y_train, X_test, y_test, threshold):
    #count the classified members (trianing) and non-members (validation)
    train_member_count, test_member_count = 0, 0

    #loop through the training data
    for idx in range(len(X_train)):
        sample = X_train[idx]
        distances = [sqrt(np.sum((x - sample)**2)) for x in X_train]
        distances.sort()
        avg_distance = np.mean(distances[1:K+1])  #skip 1st (distance to itself = 0)

        #if dist is below threshold, class as member
        if avg_distance < threshold:
            train_member_count += 1

    #loop through the validation data
    for idx in range(len(X_test)):
        sample = X_test[idx]
        distances = [sqrt(np.sum((x - sample)**2)) for x in X_train]
        distances.sort()
        avg_distance = np.mean(distances[:K])  #use the first k distances

        #if dist is below threshold, class as member
        if avg_distance < threshold:
            test_member_count += 1

    #attack success rate
    train_accuracy = train_member_count / len(X_train)
    test_accuracy = test_member_count / len(X_test)
    print(f"\nMembership Inference Attack Results - Threshold {threshold}:")
    print(f"Training set classified as members: {train_member_count}/{len(X_train)} ({train_accuracy:.2f})")
    print(f"Validation set classified as members: {test_member_count}/{len(X_test)} ({test_accuracy:.2f})")


The MIA uses a distance-based criterion: for each sample in the training and validation subsets, it calculates the Euclidean distance to all training points. Then, it computes the average distance to the k-nearest neighbour. If the average is less than our predefined threshold (see code below), the sample is classified as a member of the training data (else, it will be classified as member of the validation set).

Note on the threshold:
- a smaller threshold makes it more difficult to classify samples as members (increased precision, decreased recall).
- larger threshold increases the likelihood of classifying samples as members but may introduce false positives.
Here is a nice article that describes the difference between accuracy, precision, and recall: [link](https://www.evidentlyai.com/classification-metrics/accuracy-precision-recall).

In our example, we will set the threshold to 0.45 - let's see how it performs:

In [3]:
#MIA threshold
threshold = 0.45
mia(3, X_train, y_train, X_test, y_test, threshold) #display classification


Membership Inference Attack Results - Threshold 0.45:
Training set classified as members: 99/120 (0.82)
Validation set classified as members: 23/30 (0.77)


In an ideal scenario, the attack should classify most of the training samples as members and few test samples as members. High classification rates for both sets indicates a weak attack.

Our result is 0.82 and 0.77 for training and validation sets, respectively. They are both quite high, so we would need to adjust the threshold.

### The Task

Your task now is to use the digits dataset and perform the MIA attack. You might want to try various threshold values and see how that will impact the rate of classification.

Last week, we performed a data poisoning attack on a decision tree. Try to implement a data poisoning attack on the digits dataset.

Note: this is a larger dataset therefore execution time might take longer.

In [ ]:
#your code goes here
